# Fresh MRI cache pilot
CPU. Attach competition and saved notebook 01 output. Default: at most 120 studies per split, one series per study, 8 slices, 160 pixels. This is a speed baseline, not a full six-series production pipeline. Only selected slices are decoded. Save output for notebook 03.

In [ ]:
from pathlib import Path
import json, time, hashlib
import numpy as np
import pandas as pd
ROOTS = ['/kaggle/input/datasets/aymenslimani', '/kaggle/input']
def locate(name):
    hits = sorted(set(p.resolve() for root in ROOTS if Path(root).exists() for p in Path(root).rglob(name)))
    if not hits: raise FileNotFoundError(f'Attach the saved previous notebook output containing {name}')
    if len(hits)>1 and len({hashlib.sha256(p.read_bytes()).hexdigest() for p in hits})>1:
        raise ValueError(f'Conflicting inputs; select one attachment: {hits}')
    return hits[0]
TARGETS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('pydicom') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','pydicom']) # Internet required if missing
import pydicom
from PIL import Image
COMP=next((Path(p) for p in ['/kaggle/input/competitions/rsna-knee-abnormality-detection','/kaggle/input/rsna-knee-abnormality-detection'] if (Path(p)/'train.csv').exists()),None)
if COMP is None: raise FileNotFoundError('Attach competition')
manifest_path=locate('fresh_manifest.csv'); prep=manifest_path.parent
manifest=pd.read_csv(manifest_path,dtype={'StudyInstanceUID':str,'group':str})
series=pd.read_csv(COMP/'train_series.csv',dtype=str)
FOLD=0; MAX_PER_SPLIT=120; N_SLICES=8; SIZE=160
OUT=Path('/kaggle/working/rsna_fresh_cache'); OUT.mkdir(exist_ok=True)
chosen=pd.concat([manifest[manifest.fold!=FOLD].sample(n=min(MAX_PER_SPLIT,len(manifest[manifest.fold!=FOLD])),random_state=42),manifest[manifest.fold==FOLD].sample(n=min(MAX_PER_SPLIT,len(manifest[manifest.fold==FOLD])),random_state=42)])
# Prefer sagittal series when metadata is available; deterministic fallback by series UID.
series['priority']=0
if 'SeriesDescription' in series: series['priority']=~series.SeriesDescription.fillna('').str.contains('sag',case=False)
series=series.sort_values(['priority','SeriesInstanceUID']).drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID')
records=[]; errors=[]; start=time.time()
for _,row in chosen.iterrows():
    uid=row.StudyInstanceUID
    try:
        sid=series.loc[uid,'SeriesInstanceUID']; folder=COMP/'train_series'/uid/sid
        files=sorted(p for p in folder.iterdir() if p.is_file())
        headers=[]
        for p in files:
            try:
                d=pydicom.dcmread(p,stop_before_pixels=True)
                pos=getattr(d,'ImagePositionPatient',None); ori=getattr(d,'ImageOrientationPatient',None)
                z=float(np.dot(pos,np.cross(ori[:3],ori[3:]))) if pos is not None and ori is not None else float(getattr(d,'InstanceNumber',0))
                headers.append((z,str(p)))
            except Exception: pass
        headers.sort()
        if not headers: raise ValueError('No readable DICOM headers')
        anchors=np.unique(np.linspace(0,len(headers)-1,N_SLICES).round().astype(int)); images=[]
        for anchor in anchors:
            d=pydicom.dcmread(headers[anchor][1]); a=d.pixel_array.astype(np.float32)
            if a.ndim!=2: raise ValueError('Expected single-frame 2D MRI')
            a=a*float(getattr(d,'RescaleSlope',1))+float(getattr(d,'RescaleIntercept',0))
            lo,hi=np.percentile(a,[1,99]); a=np.clip((a-lo)/max(hi-lo,1e-6),0,1)
            if getattr(d,'PhotometricInterpretation','')=='MONOCHROME1': a=1-a
            images.append(np.asarray(Image.fromarray((a*255).astype('uint8')).resize((SIZE,SIZE))))
        arr=np.stack(images); np.savez_compressed(OUT/(uid+'.npz'),images=arr)
        records.append({'StudyInstanceUID':uid,'SeriesInstanceUID':sid,'file':uid+'.npz','n_slices':len(arr)})
    except Exception as exc: errors.append({'StudyInstanceUID':uid,'error':str(exc)})
    if (len(records)+len(errors))%20==0: print('Processed',len(records)+len(errors),'seconds',round(time.time()-start))
index=pd.DataFrame(records)
if index.empty: raise RuntimeError('No cache created; inspect DICOM support and input paths')
index.to_csv(OUT/'fresh_cache_index.csv',index=False)
chosen[chosen.StudyInstanceUID.isin(index.StudyInstanceUID)].to_csv(OUT/'fresh_cached_manifest.csv',index=False)
pd.DataFrame(errors,columns=['StudyInstanceUID','error']).to_csv(OUT/'decode_errors.csv',index=False)
(OUT/'fresh_cache.json').write_text(json.dumps({'fold':FOLD,'size':SIZE,'n_slices':N_SLICES,'max_per_split':MAX_PER_SPLIT,'manifest_sha256':hashlib.sha256(manifest_path.read_bytes()).hexdigest(),'preparation':json.loads((prep/'fresh_preparation.json').read_text()),'recipe':'one series, geometric slice ordering, percentile full FOV','seconds':time.time()-start},indent=2))
print('Cached',len(index),'failed',len(errors),'seconds',round(time.time()-start), 'Saved',OUT)

In [ ]:
import matplotlib.pyplot as plt
a=np.load(OUT/index.iloc[0]['file'])['images']
fig,axes=plt.subplots(1,len(a),figsize=(16,3))
for ax,img in zip(np.atleast_1d(axes),a): ax.imshow(img,cmap='gray'); ax.axis('off')
plt.show()
print('Inspect montage and decode_errors.csv before training; this pilot omits multi-series information.')